# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all `RecordSet`s and their `@id`s, then display their fields and columns (using `@id` as required for referencing).

In [ ]:
# List all record sets and their structure (by @id)

record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    # Croissant 1.0 uses 'recordSet' (not necessarily plural)
    record_sets = metadata.recordSet

if not record_sets:
    # Try to auto-detect record sets from metadata (sometimes it's under Dataset.records)
    record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        # Each rs is likely a string @id or a Class
        rs_meta = dataset.record_sets[rs] if isinstance(rs, str) else rs
        print(f"- Record Set @id: {rs}")
        
        # Display fields and columns with their @id
        if hasattr(rs_meta, 'fields'):
            print("    Fields:")
            for field in rs_meta.fields:
                print(f"     - Field @id: {getattr(field, '@id', getattr(field, 'id', field))}  Name: {getattr(field, 'name', '')}")
        if hasattr(rs_meta, 'columns'):
            print("    Columns:")
            for col in rs_meta.columns:
                print(f"     - Column @id: {getattr(col, '@id', getattr(col, 'id', col))}  Name: {getattr(col, 'name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Update the list of record set @ids based on the previous cell output.
# For this dataset, assume we found a main record set:

main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # This is the dataset's main DataDownload object

# Build a list for possible future scaling to more tables
record_sets_ids = [main_record_set_id]

dataframes = {}
for record_set in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set])} records from record set {record_set}")
        print(f"Fields: {dataframes[record_set].columns.tolist()}")
    except Exception as e:
        print(f"Failed to load records for Record Set {record_set}: {e}")

# Preview the main table
if main_record_set_id in dataframes:
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Tip:** For this mass spectrometry dataset, likely numeric fields include e.g. 'MW', 'peptide_counts', 'abundance', etc. Use the exact column names or field `@id`s as displayed above.

In [ ]:
# For demonstration, select common mass spec fields: MW (molecular weight), coverage, or peptide counts (update if different after inspecting column names).

df = dataframes.get(main_record_set_id)
if df is not None:
    print("Available columns for EDA:", df.columns.tolist())
    
    # Select a field: e.g. molecular weight
    numeric_field = None
    for col in df.columns:
        if 'mw' in col.lower() or 'molecular_weight' in col.lower():
            numeric_field = col
            break
    if not numeric_field:
        # fallback, pick first numeric-looking column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    assert numeric_field, "No numeric field found to demonstrate. Please adjust field selection."
    
    # Example: filter to high MW proteins (threshold arbitrary)
    threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Demonstrate grouping by a string/categorical field (e.g. description, if present)
    group_field = None
    for col in df.columns:
        if 'description' in col.lower() or 'class' in col.lower() or 'group' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll show distributions for the selected numeric field and a scatter plot if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Scatter plot (if there is a 2nd numeric field)
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field]
    if len(numeric_cols):
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_field], y=df[numeric_cols[0]])
        plt.xlabel(numeric_field)
        plt.ylabel(numeric_cols[0])
        plt.title(f"{numeric_field} vs {numeric_cols[0]}")
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the dataset and explored its main record set and fields using their `@id`s.
- Extracted tabular records and performed basic exploratory analyses on the main quantitative fields.
- Visualized field distributions and relationships in the data.

The dataset is ready for further bioinformatic, proteomic, or statistical analysis using familiar Python data science tools.

> **Tip:** For production or research, always check metadata and schema fields with their `@id`s and documentation for up-to-date field names and usage.